# Markowitz Mean-Variance Portfolio Optimization

This notebook implements the classical **Markowitz Mean-Variance Optimization** framework as a baseline for the RL-based portfolio optimization project.

### Goals
1. Download historical price data for a set of assets.
2. Compute expected returns and the covariance matrix.
3. Solve the quadratic program to find the efficient frontier.
4. Identify the **minimum-variance portfolio** and the **maximum-Sharpe-ratio portfolio**.
5. Visualize the efficient frontier.

### References
- Markowitz, H. (1952). *Portfolio Selection*. The Journal of Finance, 7(1), 77–91.
- [Portfolio Optimization Book (Roncalli)](https://portfoliooptimizationbook.com/portfolio-optimization-book.pdf)

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import scipy.optimize as sco

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

## 2. Download Price Data

In [ ]:
# Define assets and time period
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'JPM', 'GS', 'XOM', 'JNJ']
START_DATE = '2018-01-01'
END_DATE   = '2023-12-31'

# Download adjusted closing prices
prices = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)['Close']
prices.dropna(inplace=True)

print(f'Downloaded {len(prices)} trading days for {len(TICKERS)} assets')
prices.tail()

## 3. Compute Returns, Expected Returns, and Covariance

In [ ]:
# Daily log returns
returns = np.log(prices / prices.shift(1)).dropna()

# Annualized statistics (252 trading days)
TRADING_DAYS = 252
mu = returns.mean() * TRADING_DAYS          # annualised expected returns (vector)
Sigma = returns.cov() * TRADING_DAYS        # annualised covariance matrix

print('Annualized expected returns:')
print(mu.round(4))
print('\nCovariance matrix (first 4 assets):')
print(Sigma.iloc[:4, :4].round(4))

## 4. Portfolio Statistics Helper Functions

In [ ]:
def portfolio_return(weights, mu):
    """Expected annualised portfolio return."""
    return weights @ mu


def portfolio_volatility(weights, Sigma):
    """Annualized portfolio standard deviation."""
    return np.sqrt(weights @ Sigma @ weights)


def neg_sharpe(weights, mu, Sigma, rf=0.0):
    """Negative Sharpe ratio (for minimisation)."""
    ret = portfolio_return(weights, mu)
    vol = portfolio_volatility(weights, Sigma)
    return -(ret - rf) / vol

## 5. Efficient Frontier via Monte Carlo Simulation

In [ ]:
n_assets = len(TICKERS)
n_portfolios = 5000
np.random.seed(42)

results = np.zeros((3, n_portfolios))
weights_record = []

for i in range(n_portfolios):
    w = np.random.dirichlet(np.ones(n_assets))   # random weights summing to 1
    results[0, i] = portfolio_return(w, mu.values)
    results[1, i] = portfolio_volatility(w, Sigma.values)
    results[2, i] = (results[0, i]) / results[1, i]   # Sharpe (rf=0)
    weights_record.append(w)

# Plot
fig, ax = plt.subplots()
sc = ax.scatter(results[1], results[0], c=results[2], cmap='viridis', alpha=0.5, s=10)
plt.colorbar(sc, label='Sharpe Ratio')
ax.set_xlabel('Annualized Volatility')
ax.set_ylabel('Annualized Return')
ax.set_title('Monte Carlo Efficient Frontier')
plt.tight_layout()
plt.show()

## 6. Optimal Portfolios via Numerical Optimisation

In [ ]:
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
bounds = tuple((0, 1) for _ in range(n_assets))
w0 = np.ones(n_assets) / n_assets   # equal-weight starting point

# Maximum-Sharpe portfolio
result_sharpe = sco.minimize(
    neg_sharpe, w0, args=(mu.values, Sigma.values),
    method='SLSQP', bounds=bounds, constraints=constraints
)
w_max_sharpe = result_sharpe.x

# Minimum-variance portfolio
result_minvar = sco.minimize(
    portfolio_volatility, w0, args=(Sigma.values,),
    method='SLSQP', bounds=bounds, constraints=constraints
)
w_min_var = result_minvar.x

print('=== Maximum Sharpe Portfolio ===')
for ticker, w in zip(TICKERS, w_max_sharpe):
    print(f'  {ticker}: {w:.2%}')
print(f'  Return: {portfolio_return(w_max_sharpe, mu.values):.2%}')
print(f'  Volatility: {portfolio_volatility(w_max_sharpe, Sigma.values):.2%}')
print(f'  Sharpe: {-neg_sharpe(w_max_sharpe, mu.values, Sigma.values):.3f}')

print('\n=== Minimum Variance Portfolio ===')
for ticker, w in zip(TICKERS, w_min_var):
    print(f'  {ticker}: {w:.2%}')
print(f'  Return: {portfolio_return(w_min_var, mu.values):.2%}')
print(f'  Volatility: {portfolio_volatility(w_min_var, Sigma.values):.2%}')
print(f'  Sharpe: {-neg_sharpe(w_min_var, mu.values, Sigma.values):.3f}')

## 7. Next Steps

- Implement the **Black-Litterman** model (see `black_litterman.ipynb`) to incorporate market equilibrium views.
- Use these classical baselines as benchmark comparison points for the RL agents.
- Evaluate on rolling out-of-sample windows to measure real-world performance.